# 전이 데이터셋 생성

`two_visit_dataset.json`과 `multi_visit_dataset.json`을 통합하고,
검진 간격 필터를 적용한 뒤 `adoc_v1.total_checkups.json`의 원본 검진 필드를 결합하여
**pre-diabetes** / **diabetes** 학습용 데이터셋을 생성합니다.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from core.dataset_builder import TransitionDatasetBuilder

OUTPUT_DIR = Path("../outputs/260526_create_dataset")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EDA_OUTPUT_DIR = Path("../outputs/260522_EDA")

# current_checkup_date → future_checkup_date 간격 상한 (년)
MAX_INTERVAL_YEARS = 3.0

EXCLUDE_USER_KEYS = [
    "INVALID_RESULT",
]

## 1. 데이터셋 생성

- `two_visit_dataset.json` (2회 수검) + `multi_visit_dataset.json` (3회+ 수검) 통합
- 검진 간격이 `MAX_INTERVAL_YEARS` 이내인 수검자만 포함
- `detail_infos`는 `small_checkup_name` → `value` 형태로 컬럼 펼침
- `record_key`, `dataset`, `source`, `_id`, `created_at`, `updated_at`, `order_key`, `center_code`, `center_name`, `target_name`, `checkup_type` 제외
- `pre_diabetes_dataset.xlsx`, `diabetes_dataset.xlsx`로 저장

In [2]:
builder = TransitionDatasetBuilder(
    EDA_OUTPUT_DIR / "two_visit_dataset.json",
    EDA_OUTPUT_DIR / "multi_visit_dataset.json",
    max_interval_years=MAX_INTERVAL_YEARS,
    exclude_user_keys=EXCLUDE_USER_KEYS,
)

datasets = builder.build()
saved_paths = builder.export(OUTPUT_DIR)

Saved: ../outputs/260526_create_dataset/pre_diabetes_dataset.xlsx  (3,113명)
Saved: ../outputs/260526_create_dataset/diabetes_dataset.xlsx  (1,458명)


## 2. 요약

In [3]:
summary = builder.summary()
summary["mean_interval_days"] = summary["mean_interval_days"].round(1)
summary

,dataset,label,source,count,mean_interval_days
0,diabetes,0,2회,918,436.7
1,diabetes,0,3회+,469,379.2
2,diabetes,1,2회,46,468.3
3,diabetes,1,3회+,25,415.6
4,pre-diabetes,0,2회,1865,471.0
5,pre-diabetes,0,3회+,565,390.9
6,pre-diabetes,1,2회,400,470.7
7,pre-diabetes,1,3회+,283,392.1


In [5]:
pre_df = builder.to_dataframe("pre-diabetes")
print("pre-diabetes shape:", pre_df.shape)
print("columns (first 20):", list(pre_df.columns[:20]))

pre-diabetes shape: (3113, 400)
columns (first 20): ['record_key', 'user_key', 'current_checkup_date', 'future_checkup_date', 'dataset', 'label', 'transition', 'selected_transition', 'full_transition', 'source', 'interval_days', '_id', 'created_at', 'updated_at', 'order_key', 'center_code', 'center_name', 'target_name', 'checkup_date', 'checkup_type']
